# GraphLabelRepairEstimator on ZINC

This notebook trains `GraphLabelRepairEstimator` on ZINC molecular graphs, corrupts a user-defined fraction of held-out node and edge labels, repairs the corrupted graphs, and reports precision, recall, and F1 for node labels, edge labels, and the combined repair task.

In [ ]:
from pathlib import Path
import runpy

BOOTSTRAP_CANDIDATES = (
    "notebooks/_bootstrap.py",
    "abstractgraph/notebooks/_bootstrap.py",
    "abstractgraph-ml/notebooks/_bootstrap.py",
    "abstractgraph-generative/notebooks/_bootstrap.py",
    "abstractgraph-graphicalizer/notebooks/_bootstrap.py",
)

_bootstrap_path = next(
    (
        candidate / relative
        for candidate in (Path.cwd(), *Path.cwd().parents)
        for relative in BOOTSTRAP_CANDIDATES
        if (candidate / relative).exists()
    ),
    None,
)
if _bootstrap_path is None:
    raise FileNotFoundError("Could not locate ecosystem notebooks/_bootstrap.py")

_bootstrap = runpy.run_path(str(_bootstrap_path))
repo_root = _bootstrap["repo_root"]
workspace_root = _bootstrap["workspace_root"]

print("repo_root:", repo_root)
print("workspace_root:", workspace_root)

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = 'retina'
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from abstractgraph.operators import neighborhood
from abstractgraph.vectorize import AbstractGraphTransformer
from abstractgraph_ml.estimators import GraphEstimator, GraphLabelRepairEstimator
from abstractgraph_graphicalizer.chem import ZINCLoader

## Parameters

Increase `dataset_limit`, `train_size`, `test_size`, or `nbits` for a stronger run. The defaults are intentionally modest because the repair estimator creates one masked graph per repaired node and edge.

In [ ]:
dataset_name = "zinc_250k"
dataset_limit = 2000
max_node_count = 20

train_size = 1500
test_size = 100
corruption_fraction = 0.10

query_label = "?"
nbits = 14
random_state = 0

decomposition_function = neighborhood(radius=(0, 3))
classifier = RandomForestClassifier(
    n_estimators=200,
    min_samples_leaf=1,
    class_weight="balanced_subsample",
    n_jobs=-1,
    random_state=random_state,
)

## Load ZINC molecular graphs

In [ ]:
candidate_zinc_roots = [
    workspace_root / "abstractgraph-graphicalizer" / "data-local" / "ZINC",
    workspace_root / "abstractgraph-graphicalizer" / "data" / "ZINC",
]
zinc_root = next((path for path in candidate_zinc_roots if path.exists()), candidate_zinc_roots[-1])

loader = ZINCLoader(zinc_root, on_error="skip")
graphs, metadata = loader.load(
    dataset_name,
    limit=dataset_limit,
    max_node_count=max_node_count,
)

print("root:", loader.root)
print("dataset_name:", dataset_name)
print("graphs loaded:", len(graphs))
display(metadata.head())

## Train/test split

In [ ]:
if train_size + test_size > len(graphs):
    raise ValueError(
        f"Need at least train_size + test_size graphs, got {len(graphs)}. "
        "Increase dataset_limit or reduce train_size/test_size."
    )

indices = np.arange(len(graphs))
train_indices, test_indices = train_test_split(
    indices,
    train_size=train_size,
    test_size=test_size,
    random_state=random_state,
    shuffle=True,
)

train_graphs = [graphs[idx] for idx in train_indices]
test_graphs = [graphs[idx] for idx in test_indices]

print("train graphs:", len(train_graphs))
print("test graphs:", len(test_graphs))

## Fit label repair estimator

In [ ]:
transformer = AbstractGraphTransformer(
    nbits=nbits,
    decomposition_function=decomposition_function,
    return_dense=False,
    n_jobs=-1,
)

base_graph_estimator = GraphEstimator(
    transformer=transformer,
    estimator=classifier,
    manifold=None,
    n_selected_features=None,
)

repair_estimator = GraphLabelRepairEstimator(
    graph_estimator=base_graph_estimator,
    query_label=query_label,
)

repair_estimator.fit(train_graphs)

print("node classifier fitted:", repair_estimator.node_graph_estimator_ is not None)
print("edge classifier fitted:", repair_estimator.edge_graph_estimator_ is not None)

## Corrupt held-out labels

The corruption step samples exactly `round(corruption_fraction * eligible_count)` node positions and edge positions. Replacement labels are drawn from the training label vocabulary with the current true label excluded, so a corrupted label is never identical to the original label.

In [ ]:
def iter_edge_refs(graph):
    if graph.is_multigraph():
        for u, v, key, data in graph.edges(keys=True, data=True):
            yield (u, v, key), data
    else:
        for u, v, data in graph.edges(data=True):
            yield (u, v, None), data


def set_edge_label(graph, edge_ref, label):
    u, v, key = edge_ref
    if graph.is_multigraph():
        graph.edges[u, v, key]["label"] = label
    else:
        graph.edges[u, v]["label"] = label


def get_edge_label(graph, edge_ref):
    u, v, key = edge_ref
    if graph.is_multigraph():
        return graph.edges[u, v, key].get("label")
    return graph.edges[u, v].get("label")


def label_vocabulary(graphs, kind):
    labels = set()
    if kind == "node":
        for graph in graphs:
            labels.update(data["label"] for _, data in graph.nodes(data=True) if "label" in data)
    elif kind == "edge":
        for graph in graphs:
            labels.update(data["label"] for _, data in iter_edge_refs(graph) if "label" in data)
    else:
        raise ValueError("kind must be 'node' or 'edge'")
    return sorted(labels, key=str)


def choose_replacement(label, vocabulary, rng):
    candidates = [candidate for candidate in vocabulary if candidate != label]
    if not candidates:
        return None
    return candidates[int(rng.integers(len(candidates)))]


def corrupt_graph_labels(graphs, fraction, *, rng, node_vocabulary, edge_vocabulary):
    corrupted_graphs = [graph.copy() for graph in graphs]
    node_positions = []
    edge_positions = []

    for graph_idx, graph in enumerate(graphs):
        for node, data in graph.nodes(data=True):
            if "label" in data and any(candidate != data["label"] for candidate in node_vocabulary):
                node_positions.append((graph_idx, node, data["label"]))
        for edge_ref, data in iter_edge_refs(graph):
            if "label" in data and any(candidate != data["label"] for candidate in edge_vocabulary):
                edge_positions.append((graph_idx, edge_ref, data["label"]))

    records = []
    for kind, positions, vocabulary in [
        ("node", node_positions, node_vocabulary),
        ("edge", edge_positions, edge_vocabulary),
    ]:
        n_corrupt = int(round(float(fraction) * len(positions)))
        if n_corrupt == 0:
            continue
        sampled_indices = rng.choice(len(positions), size=n_corrupt, replace=False)
        for sampled_index in sampled_indices:
            graph_idx, item_ref, true_label = positions[int(sampled_index)]
            corrupted_label = choose_replacement(true_label, vocabulary, rng)
            if corrupted_label is None:
                continue
            if kind == "node":
                corrupted_graphs[graph_idx].nodes[item_ref]["label"] = corrupted_label
            else:
                set_edge_label(corrupted_graphs[graph_idx], item_ref, corrupted_label)
            records.append(
                {
                    "graph_index": graph_idx,
                    "kind": kind,
                    "item_ref": item_ref,
                    "true_label": true_label,
                    "corrupted_label": corrupted_label,
                }
            )

    columns = ["graph_index", "kind", "item_ref", "true_label", "corrupted_label"]
    return corrupted_graphs, pd.DataFrame(records, columns=columns)

In [ ]:
rng = np.random.default_rng(random_state)

node_vocab = label_vocabulary(train_graphs, "node")
edge_vocab = label_vocabulary(train_graphs, "edge")

corrupted_test_graphs, corruption_df = corrupt_graph_labels(
    test_graphs,
    corruption_fraction,
    rng=rng,
    node_vocabulary=node_vocab,
    edge_vocabulary=edge_vocab,
)

print("node label vocabulary:", node_vocab)
print("edge label vocabulary:", edge_vocab)
display(corruption_df.groupby("kind").size().rename("n_corrupted").to_frame())
display(corruption_df.head())

## Repair and evaluate

In [ ]:
repaired_test_graphs = repair_estimator.transform(corrupted_test_graphs)


def repaired_label(graphs, record):
    graph = graphs[int(record["graph_index"])]
    if record["kind"] == "node":
        return graph.nodes[record["item_ref"]].get("label")
    return get_edge_label(graph, record["item_ref"])


eval_df = corruption_df.copy()
eval_df["repaired_label"] = [repaired_label(repaired_test_graphs, record) for record in eval_df.to_dict("records")]
eval_df["correct"] = eval_df["true_label"] == eval_df["repaired_label"]

display(eval_df.head())

In [ ]:
def metric_row(frame, label):
    if frame.empty:
        return {
            "label_set": label,
            "support": 0,
            "accuracy": np.nan,
            "precision_micro": np.nan,
            "recall_micro": np.nan,
            "f1_micro": np.nan,
            "precision_macro": np.nan,
            "recall_macro": np.nan,
            "f1_macro": np.nan,
        }
    y_true = frame["true_label"].to_numpy(dtype=object)
    y_pred = frame["repaired_label"].to_numpy(dtype=object)
    p_micro, r_micro, f_micro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="micro",
        zero_division=0,
    )
    p_macro, r_macro, f_macro, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )
    return {
        "label_set": label,
        "support": len(frame),
        "accuracy": float(np.mean(y_true == y_pred)),
        "precision_micro": float(p_micro),
        "recall_micro": float(r_micro),
        "f1_micro": float(f_micro),
        "precision_macro": float(p_macro),
        "recall_macro": float(r_macro),
        "f1_macro": float(f_macro),
    }


summary = pd.DataFrame(
    [
        metric_row(eval_df[eval_df["kind"] == "node"], "node"),
        metric_row(eval_df[eval_df["kind"] == "edge"], "edge"),
        metric_row(eval_df, "total"),
    ]
)

display(summary)

In [ ]:
mismatches = eval_df.loc[~eval_df["correct"]].copy()
print("mismatches:", len(mismatches))
display(mismatches.head(20))